# COMP 469 Lab 1: Vacuum World Agents

**Topic:** AI Agents and Foundations  
**Goal:** Implement and compare reflex, model-based, and goal-based agents in a simple grid-world vacuum environment.

This notebook includes:
- A simple `VacuumEnvironment`
- `ReflexVacuumAgent`
- `ModelBasedVacuumAgent`
- `GoalBasedVacuumAgent`
- Simulations over different environment configurations
- Performance visualizations
- A reflection template


In [ ]:
import random
from collections import deque
import matplotlib.pyplot as plt

random.seed(469)


## 1. Vacuum World Environment

The environment is a rectangular grid. Each square may be clean or dirty. The agent can move in four directions or suck dirt from its current square.

A simple performance score is used:
- `+10` for successfully cleaning a dirty square
- `-1` for each movement/action cost
- `-2` for bumping into a wall or obstacle
- `+1` for each clean square at each time step


In [ ]:
class VacuumEnvironment:
    def __init__(self, width=5, height=5, dirt_prob=0.3, obstacle_prob=0.0, start=(0, 0)):
        self.width = width
        self.height = height
        self.start = start
        self.agent_location = start
        self.status = {}
        self.obstacles = set()
        self.score = 0
        self.cleaned_count = 0
        self.steps = 0

        for x in range(width):
            for y in range(height):
                if (x, y) != start and random.random() < obstacle_prob:
                    self.obstacles.add((x, y))

        for x in range(width):
            for y in range(height):
                if (x, y) in self.obstacles:
                    self.status[(x, y)] = 'Obstacle'
                else:
                    self.status[(x, y)] = 'Dirty' if random.random() < dirt_prob else 'Clean'

        self.status[start] = 'Clean'

    def in_bounds(self, location):
        x, y = location
        return 0 <= x < self.width and 0 <= y < self.height

    def is_accessible(self, location):
        return self.in_bounds(location) and location not in self.obstacles

    def get_percept(self):
        return {
            'location': self.agent_location,
            'status': self.status[self.agent_location]
        }

    def get_neighbors(self, location):
        x, y = location
        candidates = {
            'Up': (x, y - 1),
            'Down': (x, y + 1),
            'Left': (x - 1, y),
            'Right': (x + 1, y)
        }
        return {action: loc for action, loc in candidates.items() if self.is_accessible(loc)}

    def execute_action(self, action):
        self.steps += 1
        self.score -= 1

        if action == 'Suck':
            if self.status[self.agent_location] == 'Dirty':
                self.status[self.agent_location] = 'Clean'
                self.score += 10
                self.cleaned_count += 1
            return

        moves = {
            'Up': (0, -1),
            'Down': (0, 1),
            'Left': (-1, 0),
            'Right': (1, 0)
        }

        if action in moves:
            dx, dy = moves[action]
            x, y = self.agent_location
            new_location = (x + dx, y + dy)
            if self.is_accessible(new_location):
                self.agent_location = new_location
            else:
                self.score -= 2

    def count_clean_squares(self):
        return sum(1 for loc, status in self.status.items() if status == 'Clean' and loc not in self.obstacles)

    def count_dirty_squares(self):
        return sum(1 for status in self.status.values() if status == 'Dirty')

    def is_done(self):
        return self.count_dirty_squares() == 0

    def display(self):
        for y in range(self.height):
            row = []
            for x in range(self.width):
                loc = (x, y)
                if loc == self.agent_location:
                    row.append('A')
                elif loc in self.obstacles:
                    row.append('#')
                elif self.status[loc] == 'Dirty':
                    row.append('D')
                else:
                    row.append('.')
            print(' '.join(row))
        print(f'Score: {self.score}, Dirty left: {self.count_dirty_squares()}')


## 2. Agent Classes

Each agent receives a percept and returns an action.

- **Reflex agent:** reacts only to the current percept.
- **Model-based agent:** remembers visited locations and uses that internal state to avoid repeating unproductive moves.
- **Goal-based agent:** plans a path toward known dirty squares using breadth-first search.


In [ ]:
class Agent:
    def choose_action(self, percept, environment):
        raise NotImplementedError


class ReflexVacuumAgent(Agent):
    def choose_action(self, percept, environment):
        if percept['status'] == 'Dirty':
            return 'Suck'
        return random.choice(['Up', 'Down', 'Left', 'Right'])


class ModelBasedVacuumAgent(Agent):
    def __init__(self):
        self.visited = set()

    def choose_action(self, percept, environment):
        location = percept['location']
        self.visited.add(location)

        if percept['status'] == 'Dirty':
            return 'Suck'

        neighbors = environment.get_neighbors(location)
        unvisited_actions = [action for action, loc in neighbors.items() if loc not in self.visited]

        if unvisited_actions:
            return random.choice(unvisited_actions)

        if neighbors:
            return random.choice(list(neighbors.keys()))

        return 'NoOp'


class GoalBasedVacuumAgent(Agent):
    def choose_action(self, percept, environment):
        location = percept['location']

        if percept['status'] == 'Dirty':
            return 'Suck'

        dirty_locations = [loc for loc, status in environment.status.items() if status == 'Dirty']

        if not dirty_locations:
            return 'NoOp'

        path = self.bfs_to_nearest_dirty(location, dirty_locations, environment)

        if path:
            return path[0]

        return random.choice(['Up', 'Down', 'Left', 'Right'])

    def bfs_to_nearest_dirty(self, start, dirty_locations, environment):
        dirty_set = set(dirty_locations)
        queue = deque([(start, [])])
        visited = {start}

        while queue:
            location, path = queue.popleft()

            if location in dirty_set:
                return path

            for action, neighbor in environment.get_neighbors(location).items():
                if neighbor not in visited:
                    visited.add(neighbor)
                    queue.append((neighbor, path + [action]))

        return []


## 3. Simulation Runner

The simulation runs an agent in an environment for a fixed number of steps or until all dirt is cleaned.


In [ ]:
def run_simulation(agent_class, width=5, height=5, dirt_prob=0.3, obstacle_prob=0.0, max_steps=100, seed=None):
    if seed is not None:
        random.seed(seed)

    env = VacuumEnvironment(width=width, height=height, dirt_prob=dirt_prob, obstacle_prob=obstacle_prob)
    agent = agent_class()
    score_history = []
    dirty_history = []

    for _ in range(max_steps):
        env.score += env.count_clean_squares()
        percept = env.get_percept()
        action = agent.choose_action(percept, env)

        if action == 'NoOp':
            break

        env.execute_action(action)
        score_history.append(env.score)
        dirty_history.append(env.count_dirty_squares())

        if env.is_done():
            break

    return {
        'agent': agent_class.__name__,
        'score': env.score,
        'steps': env.steps,
        'cleaned': env.cleaned_count,
        'dirty_left': env.count_dirty_squares(),
        'score_history': score_history,
        'dirty_history': dirty_history
    }


## 4. Run One Example


In [ ]:
agents = [ReflexVacuumAgent, ModelBasedVacuumAgent, GoalBasedVacuumAgent]

for agent in agents:
    result = run_simulation(agent, width=5, height=5, dirt_prob=0.3, obstacle_prob=0.1, max_steps=100, seed=469)
    print(result['agent'], result)


## 5. Compare Agents Across Multiple Configurations

We test each agent across different dirt probabilities and obstacle probabilities. Each configuration runs multiple trials to reduce the effect of randomness.


In [ ]:
def compare_agents(trials=30):
    configurations = [
        {'name': 'Easy: low dirt, no obstacles', 'dirt_prob': 0.2, 'obstacle_prob': 0.0},
        {'name': 'Medium: moderate dirt, few obstacles', 'dirt_prob': 0.3, 'obstacle_prob': 0.1},
        {'name': 'Hard: high dirt, more obstacles', 'dirt_prob': 0.5, 'obstacle_prob': 0.2},
    ]

    all_results = []

    for config in configurations:
        for agent in agents:
            scores = []
            steps = []
            dirty_left = []

            for trial in range(trials):
                result = run_simulation(
                    agent,
                    width=5,
                    height=5,
                    dirt_prob=config['dirt_prob'],
                    obstacle_prob=config['obstacle_prob'],
                    max_steps=100,
                    seed=1000 + trial
                )
                scores.append(result['score'])
                steps.append(result['steps'])
                dirty_left.append(result['dirty_left'])

            all_results.append({
                'configuration': config['name'],
                'agent': agent.__name__,
                'avg_score': sum(scores) / trials,
                'avg_steps': sum(steps) / trials,
                'avg_dirty_left': sum(dirty_left) / trials
            })

    return all_results

results = compare_agents(trials=30)

for row in results:
    print(row)


## 6. Plot Results


In [ ]:
def plot_metric(results, metric, title, ylabel):
    labels = [f"{r['configuration']}
{r['agent'].replace('VacuumAgent', '').replace('Agent', '')}" for r in results]
    values = [r[metric] for r in results]

    plt.figure(figsize=(14, 5))
    plt.bar(labels, values)
    plt.xticks(rotation=45, ha='right')
    plt.title(title)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.show()

plot_metric(results, 'avg_score', 'Average Score by Agent and Environment', 'Average Score')
plot_metric(results, 'avg_dirty_left', 'Average Dirt Left by Agent and Environment', 'Average Dirty Squares Left')


## 7. Optional: Visualize One Environment


In [ ]:
env = VacuumEnvironment(width=5, height=5, dirt_prob=0.4, obstacle_prob=0.1, start=(0, 0))
env.display()


## 8. One-Page Reflection

Write a one-page reflection answering the following prompts:

1. **Which agent performed best overall?**  
   In my simulations, the goal-based agent generally performed best because it used knowledge of the environment to plan paths toward dirty squares rather than wandering randomly.

2. **How did the reflex agent behave?**  
   The reflex agent cleaned dirt when it happened to be on a dirty square, but when the square was clean it moved randomly. This made it simple but inefficient, especially in larger or obstacle-filled environments.

3. **How did the model-based agent improve on the reflex agent?**  
   The model-based agent remembered visited squares, which helped it avoid repeating the same locations too often. This gave it better coverage than the reflex agent, but it still did not plan directly toward remaining dirt.

4. **Why did the goal-based agent often perform best?**  
   The goal-based agent had an explicit goal: find and clean dirty squares. By using breadth-first search, it selected actions that moved it toward the nearest dirty square. This reduced wasted movement and usually produced higher scores.

5. **What does this lab show about rational agents?**  
   This lab shows that agent performance depends on the environment, the agent's percepts, its internal knowledge, and the performance measure. A more informed agent can behave more rationally when it has useful knowledge and a clear goal.
